In [1]:
import wntr
import numpy as np
import pandas as pd

In [2]:
from wntr.epanet.util import to_si, FlowUnits, HydParam

wn = wntr.network.WaterNetworkModel(r"C:\Data\Project\Phase 0 (Toy)\EPANET\baby_network.inp")
EMITTER_C_TO_SI = to_si(FlowUnits.GPM, 1.0, HydParam.EmitterCoeff)
GPM_TO_M3S = 0.0000630902
FT_TO_M = 0.3048


def run_simulation(
    demand_j1_gpm,
    demand_j3_gpm,
    reservoir_head_ft,
    leak_node=None,
    emitter_coeff=0
):
    # Demands
    wn.get_node("1").demand_timeseries_list[0].base_value = (
        demand_j1_gpm * GPM_TO_M3S
    )

    wn.get_node("3").demand_timeseries_list[0].base_value = (
        demand_j3_gpm * GPM_TO_M3S
    )

    # Reservoir head
    wn.get_node("2").base_head = reservoir_head_ft * FT_TO_M

    # Remove existing leaks
    for node in wn.junction_name_list:
        wn.get_node(node).emitter_coefficient = 0

    # Add leak
    if leak_node is not None:
        wn.get_node(leak_node).emitter_coefficient = (
            emitter_coeff * EMITTER_C_TO_SI
        )

    # Run EPANET
    sim = wntr.sim.EpanetSimulator(wn)
    results = sim.run_sim()

    # Measurements
    P = results.node["pressure"].iloc[0]
    Q = results.link["flowrate"].iloc[0]

    x = [
        P["1"],
        P["3"],
        Q["1"],
        Q["2"]
    ]

    y = int(leak_node is not None)

    return x, y

In [3]:
rng = np.random.default_rng(42)

def sample_positive_normal(mean, std):
    return max(rng.normal(mean, std), 0.01)


N = 100000
data = []

for _ in range(N):

    # Operating conditions
    demand_j1 = sample_positive_normal(10, 2)
    demand_j3 = sample_positive_normal(10, 2)
    reservoir_head = sample_positive_normal(100, 10)

    # Leak condition
    leak = rng.random() < 0.5

    if leak:
        leak_node = rng.choice(["1", "3"])

        # More samples near the difficult leak/no-leak boundary
        if rng.random() < 0.4:
            emitter_coeff = rng.uniform(0.01, 0.2)
        else:
            emitter_coeff = max(rng.normal(1, 0.25), 0.2)

    else:
        leak_node = None
        emitter_coeff = 0

    # Simulation
    x, y = run_simulation(
        demand_j1,
        demand_j3,
        reservoir_head,
        leak_node,
        emitter_coeff
    )

    data.append([
        *x,
        demand_j1,
        demand_j3,
        reservoir_head,
        leak_node,
        emitter_coeff,
        y
    ])


columns = [
    "P1", "P3", "Q1", "Q2",
    "demand_j1", "demand_j3", "reservoir_head",
    "leak_node", "emitter_coeff", "y"
]

dataset = pd.DataFrame(data, columns=columns)

print(dataset.head())
print(dataset.shape)

          P1         P3        Q1        Q2  demand_j1  demand_j3  \
0  32.766598  32.766434  0.001169  0.000500  10.609434   7.920032   
1  30.869223  30.869083  0.000851  0.000467   6.097930   7.395641   
2  33.159645  33.159470  0.001152  0.000523   9.966398   8.293912   
3  31.903759  31.903397  0.001483  0.000773  10.132061  12.254482   
4  28.403667  28.403439  0.001232  0.000608   9.900148   9.630275   

   reservoir_head leak_node  emitter_coeff  y  
0      107.504512       NaN        0.00000  0  
1      101.278404       NaN        0.00000  0  
2      108.793980       NaN        0.00000  0  
3      104.675093         1        0.16725  1  
4       93.190705       NaN        0.00000  0  
(100000, 10)


In [4]:
train_dataset = dataset.sample(frac=0.8, random_state=42)
test_dataset = dataset.drop(train_dataset.index)

train_dataset.to_csv(
    r"C:\Data\Project\Phase 0 (Toy)\toy_dataset(train).csv",
    index=False
)

test_dataset.to_csv(
    r"C:\Data\Project\Phase 0 (Toy)\toy_dataset(test).csv",
    index=False
)

print(train_dataset.shape)
print(test_dataset.shape)

(80000, 10)
(20000, 10)
